In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.silver_pm25 AS
WITH parsed AS (
    SELECT
        sensor_id,
        ingested_at,
        CAST(get_json_object(raw_json, '$.value') AS DOUBLE) AS pm25_ugm3,
        get_json_object(raw_json, '$.period.interval') AS interval_text,
        CAST(get_json_object(raw_json, '$.period.datetimeFrom.utc') AS TIMESTAMP) AS period_start_utc,
        CAST(get_json_object(raw_json, '$.period.datetimeTo.utc') AS TIMESTAMP) AS period_end_utc
    FROM workspace.delhi_air.bronze_openaq_measurements
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY sensor_id, period_start_utc
               ORDER BY ingested_at DESC
           ) AS rn
    FROM parsed
)
SELECT
    k.sensor_id,
    r.location_id,
    r.location_name,
    r.provider,
    k.period_start_utc,
    k.period_end_utc,
    k.interval_text,
    k.pm25_ugm3
FROM ranked k
JOIN workspace.delhi_air.sensor_registry r
  ON k.sensor_id = r.sensor_id
WHERE k.rn = 1
""")
print("done")